# ASR 2026 — Русские числа, инференс ансамбля

Команда **Mamarin & Solomenchuk** (Георгий Мамарин, Оксана Соломенчук).
Соревнование: [asr-2026-spoken-numbers-recognition-challenge](https://www.kaggle.com/competitions/asr-2026-spoken-numbers-recognition-challenge).

Публичный результат: **0.946 CER**, место #3 на момент отправки.

Публичный репозиторий с кодом и отчётом: <https://github.com/dmagog/aith_speech_recognirion_dz3>.

## Как устроен ноутбук

1. Скачивает исходный код модели и декодера архивом с GitHub.
2. Скачивает два чекпоинта ансамбля с GitHub Release.
3. Находит тестовый набор соревнования на `/kaggle/input/`.
4. Прогоняет ансамбль с TTA (сдвиг тона ±0.5 пт + полоса 5.5 кГц).
5. Грамматический CTC-лучевой декодер переводит слова в числа.
6. Пишет `submission.csv` в `/kaggle/working/`.

Требуется включённый интернет (Settings → «Internet: on»). Если интернет недоступен, можно положить чекпоинты в Kaggle-датасет и указать путь в ячейке с константами ниже.


## 1. Константы и импорты


In [ ]:
import io
import math
import os
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO = "dmagog/aith_speech_recognirion_dz3"
TAG = "v1.0"
REPO_ZIP_URL = f"https://github.com/{REPO}/archive/refs/heads/main.zip"
WEIGHT_URLS = {
    "best.pt":         f"https://github.com/{REPO}/releases/download/{TAG}/best.pt",
    "best_seed43.pt": f"https://github.com/{REPO}/releases/download/{TAG}/best_seed43.pt",
}

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_DIR / "asr_repo"
WEIGHTS_DIR = WORK_DIR / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)

print("python:", sys.version.split()[0])


## 2. Скачиваем код и веса с GitHub


In [ ]:
def download(url: str, dst: Path) -> None:
    if dst.exists() and dst.stat().st_size > 0:
        print(f"already present: {dst}")
        return
    print(f"downloading {url} ...")
    urllib.request.urlretrieve(url, dst)
    print(f"  saved {dst.stat().st_size/1e6:.1f} MB -> {dst}")

def find_in_kaggle_input(name: str) -> Path | None:
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    for cand in root.rglob(name):
        if cand.is_file() and cand.stat().st_size > 0:
            return cand
    return None

def fetch_with_fallback(url: str, dst: Path, filename: str) -> None:
    try:
        download(url, dst)
        return
    except Exception as e:
        print(f"download failed: {e}\n  trying Kaggle Dataset fallback...")
    local = find_in_kaggle_input(filename)
    if local is None:
        raise RuntimeError(f"{filename} не найден: ни интернет, ни /kaggle/input. Подключите Kaggle Dataset 'georgymamarin/asr-numbers-best-checkpoint' или включите интернет.")
    dst.write_bytes(local.read_bytes())
    print(f"  copied from {local} -> {dst} ({dst.stat().st_size/1e6:.1f} MB)")

# 1) Исходный код репозитория
zip_path = WORK_DIR / "repo_main.zip"
try:
    download(REPO_ZIP_URL, zip_path)
    if not REPO_DIR.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(WORK_DIR)
        extracted = next(WORK_DIR.glob("aith_speech_recognirion_dz3-*"))
        extracted.rename(REPO_DIR)
    src_root = REPO_DIR / "src"
except Exception as e:
    print(f"github fetch failed: {e}\n  ожидается, что src код доступен через Kaggle Dataset под /kaggle/input/.../src/")
    src_cand = find_in_kaggle_input("asr_numbers")
    if src_cand is None:
        raise RuntimeError("asr_numbers не найден. Включите интернет или подключите Kaggle Dataset с исходниками.")
    src_root = src_cand.parent
print("src at:", src_root)

# 2) Веса
for name, url in WEIGHT_URLS.items():
    fetch_with_fallback(url, WEIGHTS_DIR / name, name)

sys.path.insert(0, str(src_root))


## 3. Импорты модели и декодера


In [ ]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import yaml
from torch.utils.data import DataLoader
from functools import partial

from asr_numbers.dataset import NumbersDataset, collate_batch
from asr_numbers.decoder import grammar_beam_decode
from asr_numbers.model import ConvGRUCTCModel
from asr_numbers.text import best_effort_number_from_text
from asr_numbers.tta import tta_forward
from asr_numbers.vocab import WordVocabulary

TTA_VARIANTS = [
    {"pitch_semitones": 0.5},
    {"pitch_semitones": -0.5},
    {"bandwidth_cutoff_hz": 5500.0},
]

CONFIG_PATH = REPO_DIR / "configs" / "reverb_cont.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
print("model config:", {k: config["model"][k] for k in ("sample_rate", "n_mels", "encoder_dim", "hidden_size", "num_gru_layers")})


## 4. Загрузка ансамбля


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

vocab = WordVocabulary()
models = []
for name in ("best.pt", "best_seed43.pt"):
    path = WEIGHTS_DIR / name
    model = ConvGRUCTCModel(vocab_size=vocab.size, **config["model"]).to(device)
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model_state"])
    model.eval()
    models.append(model)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"loaded {name}: params={params/1e6:.2f} M")


## 5. Поиск тестового набора соревнования

Ищем `test.csv` внутри `/kaggle/input/` и используем её директорию как корень аудио.


In [ ]:
def locate_test_split():
    input_root = Path("/kaggle/input")
    candidates = sorted(input_root.rglob("test.csv"))
    for csv_path in candidates:
        frame = pd.read_csv(csv_path)
        if "filename" not in frame.columns or len(frame) == 0:
            continue
        probe = csv_path.parent / frame.iloc[0]["filename"]
        if probe.exists():
            return csv_path, csv_path.parent
    raise FileNotFoundError("Не нашли test.csv в /kaggle/input")

test_csv, audio_root = locate_test_split()
print("test_csv =", test_csv)
print("audio_root =", audio_root)
print("test rows:", len(pd.read_csv(test_csv)))


## 6. Инференс ансамбля с TTA и грамматическим лучевым декодером


In [ ]:
dataset = NumbersDataset(
    csv_path=test_csv,
    audio_root=audio_root,
    sample_rate=int(config["model"]["sample_rate"]),
    with_labels=False,
)
loader = DataLoader(
    dataset, batch_size=32, shuffle=False, num_workers=2,
    collate_fn=partial(collate_batch, vocab=vocab),
)
sample_rate = int(config["model"]["sample_rate"])

rows = []
with torch.no_grad():
    for step, batch in enumerate(loader, start=1):
        waveforms = batch["waveforms"].to(device)
        wl = batch["waveform_lengths"].to(device)

        per_lp, per_len = [], []
        for m in models:
            lp, ol = tta_forward(m, waveforms, wl, sample_rate, TTA_VARIANTS)
            per_lp.append(lp); per_len.append(ol)
        min_T = min(lp.size(1) for lp in per_lp)
        fused = torch.stack([lp[:, :min_T, :] for lp in per_lp], dim=0).mean(dim=0)
        out_len = torch.stack(per_len, dim=0).min(dim=0).values.clamp_max(min_T)

        sentences = grammar_beam_decode(fused.cpu(), out_len.cpu(), vocab, beam_size=16)
        for fn, s in zip(batch["filenames"], sentences):
            rows.append({"filename": fn, "transcription": str(best_effort_number_from_text(s))})
        if step % 25 == 0:
            print(f"batches={step} rows={len(rows)}")

submission = pd.DataFrame(rows, columns=["filename", "transcription"])
print("total rows:", len(submission))
submission.head()


## 7. Сохранение submission.csv


In [ ]:
out_path = WORK_DIR / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"wrote {out_path} ({out_path.stat().st_size/1024:.1f} KB)")
print(submission.sample(min(5, len(submission)), random_state=0).to_string(index=False))
